# Pecan Street preprocessing 

**Modeling assumptions:** original consumer homes without any PV readings are assigned zero PV, explicitly flagged as an assumption. Whole-home demand is signed grid import plus PV, assuming no storage/other generation. Confirm these assumptions before interpreting physical results. Prosumer PV with no observations is not fabricated.

Outputs named `*_optimization_15min.csv` contain only common valid periods for every selected household. They may be empty if a household has no recoverable grid/PV data. Use original timestamps and choose contiguous runs for storage/HEMS simulations; do not connect separated periods as consecutive steps. Appliance constraints require separate validation.


In [ ]:

# Extract configuration information from metadata.csv

metadata = pd.read_csv(METADATA_FILE, low_memory=False)

metadata["dataid"] = pd.to_numeric(
    metadata["dataid"], errors="coerce"
)

metadata = metadata[
    metadata["dataid"].isin(PARTICIPANTS)
].copy()

metadata["dataid"] = metadata["dataid"].astype(int)
metadata = metadata.set_index("dataid")

# Verify all selected households exist
missing = set(PARTICIPANTS) - set(metadata.index)
if missing:
    raise ValueError(f"Missing households: {sorted(missing)}")

if metadata.index.has_duplicates:
    raise ValueError("Duplicate household IDs in metadata")

# Helper for metadata yes/no fields
def has_yes(series):
    return series.fillna("").astype(str).str.strip().str.lower().eq("yes")

# Extract solar and grid monitoring information
pv_installed = has_yes(metadata["pv"])
solar_meter = has_yes(metadata["solar"])
grid_meter = has_yes(metadata["grid"])

# Extract battery/storage indications
storage_columns = [
    "battery1",
    "energy_storage_system",
    "energy_storage_system_l1",
    "energy_storage_system_l2"
]

storage_indicated = pd.Series(False, index=metadata.index)

for col in storage_columns:
    storage_indicated |= has_yes(metadata[col])

# Extract installed PV capacity
pv_capacity = pd.to_numeric(
    metadata["total_amount_of_pv"],
    errors="coerce"
)

# Confirm prosumers have PV and solar monitoring
CONFIRMED_PV_HOMES = set(
    metadata.index[pv_installed & solar_meter]
)

# Missing PV metadata is NOT proof of no PV
CONFIRMED_NO_PV = set()

# Grid + PV demand balance cannot be verified
# from equipment metadata alone
CONFIRMED_BALANCE_HOMES = set()

# Retain assumptions explicitly
ASSUME_ORIGINAL_CONSUMERS_NO_PV = True
USE_GRID_PLUS_PV_DEMAND = True

# Build a summary table
metadata_summary = pd.DataFrame({
    "group": [
        "prosumer" if h in PROSUMERS else "consumer"
        for h in metadata.index
    ],
    "pv_installed": pv_installed,
    "solar_meter": solar_meter,
    "grid_meter": grid_meter,
    "storage_indicated": storage_indicated,
    "pv_capacity": pv_capacity
})

metadata_summary = metadata_summary.reindex(PARTICIPANTS)

print("Confirmed PV homes:", sorted(CONFIRMED_PV_HOMES))
print("Confirmed no-PV homes:", sorted(CONFIRMED_NO_PV))
print("Confirmed balance homes:", sorted(CONFIRMED_BALANCE_HOMES))
print("Homes with storage indications:",
      metadata_summary.index[
          metadata_summary["storage_indicated"]
      ].tolist())

display(metadata_summary)


## Load and validate timestamps
The archive is not extracted automatically. Extract it separately if the CSV files are unavailable. Duplicate household/timestamp records are rejected instead of silently averaged.


In [ ]:
assert SOURCE_POWER_UNIT == 'kW', 'Adapt conversions to documented source units first.'
data = pd.read_csv(INPUT_FILE)
metadata = pd.read_csv(METADATA_FILE) if METADATA_FILE.exists() else None
required = {'dataid', 'local_15min', 'grid', 'solar'}
assert required <= set(data.columns), f'Missing columns: {required - set(data.columns)}'
data['dataid'] = pd.to_numeric(data['dataid'], errors='raise')
data = data.loc[data['dataid'].isin(PARTICIPANTS)].copy()
assert set(PARTICIPANTS) <= set(data['dataid']), 'Some selected participants are absent.'
if TIMESTAMPS_ARE_NAIVE_LOCAL:
    parsed = pd.to_datetime(data['local_15min'], errors='raise')
    data['timestamp_utc'] = parsed.dt.tz_localize(LOCAL_TZ, ambiguous='raise', nonexistent='raise').dt.tz_convert('UTC')
else:
    # Reject naive strings to avoid accidentally interpreting Austin local time as UTC.
    has_offset = data['local_15min'].astype(str).str.contains(r'(?:Z|[+-]\d{2}:?\d{2})$', regex=True)
    assert has_offset.all(), 'Naive timestamps detected: inspect source timezone and set configuration.'
    data['timestamp_utc'] = pd.to_datetime(data['local_15min'], utc=True, errors='raise')
assert data['timestamp_utc'].notna().all(), 'Missing timestamps.'
assert not data.duplicated(['dataid','timestamp_utc']).any(), 'Duplicate household/timestamp rows: investigate source.'
data = data.sort_values(['dataid','timestamp_utc'])
assert data['timestamp_utc'].eq(data['timestamp_utc'].dt.floor('15min')).all(), 'Off-grid timestamps.'
start = pd.Timestamp(STUDY_START) if STUDY_START else data['timestamp_utc'].min()
end = pd.Timestamp(STUDY_END) if STUDY_END else data['timestamp_utc'].max() + pd.Timedelta(minutes=15)
assert start.tzinfo is not None and end.tzinfo is not None
start, end = start.tz_convert('UTC'), end.tz_convert('UTC')
assert start < end and start == start.floor('15min') and end == end.floor('15min')
timeline = pd.date_range(start, end, freq='15min', inclusive='left', name='timestamp_utc')
print(f'{len(PARTICIPANTS)} homes; {len(timeline)} shared intervals; {start} to {end} (exclusive)')


In [ ]:
LOAD_CHANNELS = ['air1','air2','airwindowunit1','bathroom1','bathroom2',
 'bedroom1','bedroom2','bedroom3','bedroom4','bedroom5','car1',
 'clotheswasher1','clotheswasher_dryg1','diningroom1','dishwasher1','disposal1',
 'drye1','dryg1','freezer1','furnace1','furnace2','garage1','kitchen1','kitchen2',
 'kitchenapp1','kitchenapp2','lights_plugs1','lights_plugs2','lights_plugs3',
 'lights_plugs4','livingroom1','microwave1','office1','outsidelights_plugs1',
 'oven1','oven2','range1','refrigerator1','refrigerator2','sprinkler1',
 'utilityroom1','venthood1','waterheater1','waterheater2','winecooler1']
# Retain every available source channel, including channels omitted in the original notebook.
CHANNELS = [c for c in data.columns if c not in ['dataid','local_15min','timestamp_utc']]
for c in CHANNELS:
    data[c] = pd.to_numeric(data[c], errors='raise')
data[CHANNELS] = data[CHANNELS].replace([np.inf, -np.inf], np.nan)


## Calendar and optional prices
Prices below reproduce the original time bands with June–September as summer. Federal holidays are features only: they do not change prices without verified tariff rules. Time features use Austin local time and include UTC offset to distinguish repeated DST hours.


In [ ]:
def illustrative_tou(t):
    summer = 6 <= t.month <= 9
    if summer:
        if t.dayofweek < 5 and 14 <= t.hour < 20:
            return 0.09761
        return 0.05040 if 6 <= t.hour < 22 else 0.00493
    return 0.01201 if 6 <= t.hour < 22 else 0.00924

def add_features(frame):
    out = frame.copy()
    if out.empty:
        return out
    local = out.index.tz_convert(LOCAL_TZ)
    out['timestamp_local'] = local.astype(str)
    out['hour'] = local.hour
    out['day_of_week'] = local.dayofweek
    out['month'] = local.month
    out['utc_offset_minutes'] = [int(t.utcoffset().total_seconds()/60) for t in local]
    dates = local.tz_localize(None).normalize()
    holidays = USFederalHolidayCalendar().holidays(
        start=dates.min()-pd.Timedelta(days=7), end=dates.max()+pd.Timedelta(days=7))
    out['is_federal_holiday'] = dates.isin(holidays).astype(int)
    out['import_price_usd_per_kwh'] = [illustrative_tou(t) for t in local] if TARIFF_CONFIRMED else np.nan
    out['export_credit_usd_per_kwh'] = EXPORT_CREDIT_USD_PER_KWH if TARIFF_CONFIRMED else np.nan
    return out


In [ ]:
def impute_channels(frame):
    """Flagged offline estimates; donor statistics use observed values only."""
    result = frame.copy()
    flags = pd.DataFrame(False, index=frame.index, columns=frame.columns)
    local = frame.index.tz_convert(LOCAL_TZ)
    slot = local.hour * 4 + local.minute // 15
    weekend = local.dayofweek >= 5
    for channel in frame:
        observed = frame[channel]
        if observed.notna().sum() == 0:
            continue
        missing = observed.isna()
        # Fill only entire interior runs whose length is within the limit.
        groups = missing.ne(missing.shift()).cumsum()
        run_size = missing.groupby(groups).transform('size')
        interpolated = observed.interpolate(method='time', limit_area='inside')
        filled = observed.where(~(missing & run_size.le(MAX_SHORT_GAP)), interpolated)
        donors = pd.DataFrame({'value':observed.to_numpy(), 'slot':slot, 'weekend':weekend})
        for keys in [['slot','weekend'], ['slot']]:
            grouped = donors.groupby(keys)['value']
            medians = grouped.transform('median').set_axis(frame.index)
            counts = grouped.transform('count').set_axis(frame.index)
            filled = filled.fillna(medians.where(counts.ge(MIN_PROFILE_DONORS)))
        if observed.notna().sum() >= MIN_MEDIAN_DONORS:
            filled = filled.fillna(observed.median())
        result[channel] = filled
        flags[channel] = missing & filled.notna()
    return result, flags


In [ ]:
frames, quality, availability = {}, [], []
for home in PARTICIPANTS:
    raw = data.loc[data.dataid.eq(home)].set_index('timestamp_utc')[CHANNELS]
    aligned = raw.reindex(timeline)
    aligned, imputation_flags = impute_channels(aligned)
    out = pd.DataFrame(index=timeline)
    for channel in CHANNELS:
        out[channel + '_imputed'] = imputation_flags[channel]
    out['any_channel_imputed'] = imputation_flags.any(axis=1)
    out['dataid'] = home
    out['role'] = 'prosumer' if home in PROSUMERS else 'consumer'
    out['source_row_present'] = timeline.isin(raw.index)
    # Explicit units; voltage channels must never be summed as energy.
    for c in CHANNELS:
        suffix = '_v' if c in ['leg1v','leg2v'] else '_kw'
        out[c + suffix] = aligned[c]
    pv_channels = [c for c in ['solar','solar2'] if c in aligned and raw[c].notna().any()]
    if home in CONFIRMED_NO_PV or (ASSUME_ORIGINAL_CONSUMERS_NO_PV and home in CONSUMERS and not pv_channels):
        assert not pv_channels, f'{home}: PV measurements conflict with no-PV declaration.'
        pv = pd.Series(0.0, index=timeline)
    elif pv_channels:
        pv = aligned[pv_channels].sum(axis=1, min_count=len(pv_channels))
    else:
        pv = pd.Series(np.nan, index=timeline)
    out['pv_kw'] = pv
    out['pv_zero_assumed'] = bool(ASSUME_ORIGINAL_CONSUMERS_NO_PV and home in CONSUMERS and not pv_channels and home not in CONFIRMED_NO_PV)
    out['required_input_imputed'] = imputation_flags[['grid'] + pv_channels].any(axis=1)
    # Avoid representing an incomplete submeter sum as total house demand.
    measured_loads = [c for c in LOAD_CHANNELS if c in raw and raw[c].notna().any()]
    out['submeter_total_kw'] = aligned[measured_loads].sum(axis=1, min_count=len(measured_loads)) if measured_loads else np.nan
    out['demand_kw'] = aligned['grid'] + pv if USE_GRID_PLUS_PV_DEMAND or home in CONFIRMED_BALANCE_HOMES else np.nan
    out['balance_minus_submeters_kw'] = out['demand_kw'] - out['submeter_total_kw']
    out['negative_demand_flag'] = out['demand_kw'].lt(0)
    out['negative_pv_flag'] = out['pv_kw'].lt(0)
    out['valid_for_optimization'] = (out[['demand_kw','pv_kw','grid_kw']].notna().all(axis=1)
                                     & ~out['negative_demand_flag'] & ~out['negative_pv_flag'])
    for c in ['demand','pv','grid']:
        out[c + '_kwh'] = out[c + '_kw'] * 0.25
    frames[home] = out
    quality.append({'dataid':home, 'expected_intervals':len(timeline),
                    'missing_rows':int((~out.source_row_present).sum()),
                    'missing_grid_before':int(raw.reindex(timeline)['grid'].isna().sum()),
                    'imputed_grid':int(imputation_flags['grid'].sum()),
                    'missing_grid_after':int(out.grid_kw.isna().sum()),
                    'missing_pv':int(out.pv_kw.isna().sum()),
                    'valid_intervals':int(out.valid_for_optimization.sum())})
    for c in CHANNELS:
        availability.append({'dataid':home,'channel':c,'observed_intervals':int(raw.reindex(timeline)[c].notna().sum()),
                             'imputed_intervals':int(imputation_flags[c].sum()),
                             'all_missing_in_window':bool(aligned[c].isna().all())})
quality_report = pd.DataFrame(quality)
channel_report = pd.DataFrame(availability)
display(quality_report)


## Export 15-minute profiles and optional hourly profiles
Hourly energy requires four nonmissing quarter-hour measurements. Hourly power and voltage use means with the same completeness requirement. A shared valid-window mask is exported for market simulations; gaps are not removed or compressed. Unrecoverable values are serialized as empty CSV fields. Complete optimization exports exclude invalid common periods.


In [ ]:
def hourly_profile(out):
    h = pd.DataFrame(index=out.resample('1h').size().index)
    h['dataid'] = out['dataid'].iloc[0]
    h['role'] = out['role'].iloc[0]
    for c in out.columns:
        if c.endswith('_imputed') or c == 'pv_zero_assumed':
            h[c] = out[c].resample('1h').max()
        elif c.endswith(('_kw','_v')):
            r = out[c].resample('1h')
            h[c] = r.mean().where(r.count().eq(4))
        elif c.endswith('_kwh'):
            h[c] = out[c].resample('1h').sum(min_count=4)
    h['observed_quarters'] = out['source_row_present'].resample('1h').sum()
    h['valid_for_optimization'] = out['valid_for_optimization'].resample('1h').sum().eq(4)
    return h

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for home, out in frames.items():
    add_features(out).to_csv(OUTPUT_DIR / f'{home}_15min.csv', index=True)
    add_features(hourly_profile(out)).to_csv(OUTPUT_DIR / f'{home}_hourly.csv', index=True)
quality_report.to_csv(OUTPUT_DIR / 'quality_report.csv', index=False)
channel_report.to_csv(OUTPUT_DIR / 'channel_availability.csv', index=False)
common_valid = pd.DataFrame({str(h):f['valid_for_optimization'] for h,f in frames.items()})
common_valid['all_homes_valid'] = common_valid.all(axis=1)
common_valid.to_csv(OUTPUT_DIR / 'shared_valid_intervals.csv')
# Complete common periods, retaining real timestamps so gaps are never compressed.
for home, out in frames.items():
    complete = out.loc[common_valid['all_homes_valid']]
    add_features(complete).to_csv(OUTPUT_DIR / f'{home}_optimization_15min.csv') if len(complete) else complete.to_csv(OUTPUT_DIR / f'{home}_optimization_15min.csv')
manifest = {'input_file':str(INPUT_FILE),'source_power_unit':SOURCE_POWER_UNIT,
            'interval_minutes':15,'local_timezone':LOCAL_TZ,'study_start_utc':str(start),
            'study_end_exclusive_utc':str(end),'tariff_confirmed':TARIFF_CONFIRMED,
            'confirmed_no_pv':sorted(CONFIRMED_NO_PV),
            'confirmed_balance_homes':sorted(CONFIRMED_BALANCE_HOMES),
            'demand_definition':'signed net grid import + total measured PV; only for confirmed homes',
            'missing_policy':'short bounded interpolation; observed local-time profiles; observed median fallback; unsupported channels remain unavailable',
            'assume_original_consumers_no_pv':ASSUME_ORIGINAL_CONSUMERS_NO_PV,
            'use_grid_plus_pv_demand':USE_GRID_PLUS_PV_DEMAND,
            'imputation_mode':'offline historical; uses future observations; not forecasting-safe',
            'pandas_version':pd.__version__}
(OUTPUT_DIR / 'preprocessing_manifest.json').write_text(json.dumps(manifest, indent=2))
print(f'Wrote outputs to {OUTPUT_DIR.resolve()}')
print(f'Shared valid intervals: {int(common_valid.all_homes_valid.sum())}/{len(common_valid)}')
if not TARIFF_CONFIRMED:
    print('Tariff fields remain missing until verified. Demand uses the explicit grid-plus-PV assumption.')


## HEMS and forecasting use
Retain the appliance channels to identify flexible loads. Confirm circuit coverage and overlap before constructing fixed demand: subtract the original flexible appliance profiles from whole-home demand, then add optimized schedules. Do not assume measured power alone provides EV departure times, battery capacities, HVAC comfort limits, or appliance deadlines.

Keep optimization variables in physical units. For forecasting, split chronologically before fitting a scaler, fit on training data only, and invert predictions before calculating energy balances or costs. Any gap filling should use information available at the forecast decision time. Hourly files are optional; 15-minute profiles preserve the source resolution.
